In [20]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent
RESULTS_DIR = ROOT / "Results"

In [21]:
pycaret = pd.read_csv(RESULTS_DIR / "pycaret_results.csv")
tpot = pd.read_csv(RESULTS_DIR / "tpot_results.csv")
h2o = pd.read_csv(RESULTS_DIR / "h2o_results.csv")

In [22]:
pycaret_clean = pycaret.rename(
    columns={"search_runtime_seconds": "runtime_seconds"}
).copy()

tpot_clean = tpot.rename(
    columns={"search_runtime_seconds": "runtime_seconds"}
).copy()

h2o_clean = h2o.copy()

pycaret_clean["models_evaluated"] = pd.NA
tpot_clean["models_evaluated"] = pd.NA

In [6]:
import pandas as pd

pycaret_clean = pycaret.copy()
tpot_clean = tpot.copy()
h2o_clean = h2o.copy()

pycaret_clean = pycaret_clean.rename(
    columns={"search_runtime_seconds": "runtime_seconds"}
)

tpot_clean = tpot_clean.rename(
    columns={"search_runtime_seconds": "runtime_seconds"}
)

# Model counts were not recorded for PyCaret and TPOT
pycaret_clean["models_evaluated"] = pd.NA
tpot_clean["models_evaluated"] = pd.NA

In [23]:
common_columns = [
    "framework",
    "dataset",
    "seed",
    "accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "runtime_seconds",
    "best_model",
    "models_evaluated",
]

all_results = pd.concat(
    [
        pycaret_clean[common_columns],
        tpot_clean[common_columns],
        h2o_clean[common_columns],
    ],
    ignore_index=True,
)

all_results = all_results.sort_values(
    ["dataset", "framework", "seed"]
).reset_index(drop=True)

all_results.to_csv(
    RESULTS_DIR / "all_results.csv",
    index=False,
)

print("Total experiments:", len(all_results))
display(all_results)

Total experiments: 27


,framework,dataset,seed,accuracy,precision_macro,recall_macro,f1_macro,runtime_seconds,best_model,models_evaluated
0,H2O AutoML,breast_cancer,42,0.929825,0.919508,0.939484,0.926570,602.504012,DeepLearning_grid_1_AutoML_1_20260726_204956_m...,128
1,H2O AutoML,breast_cancer,123,0.991228,0.988372,0.993056,0.990621,602.018388,StackedEnsemble_BestOfFamily_6_AutoML_2_202607...,116
2,H2O AutoML,breast_cancer,2026,0.912281,0.902188,0.925595,0.908887,598.102286,GBM_3_AutoML_3_20260726_211003,62
3,PyCaret,breast_cancer,42,0.982456,0.981151,0.981151,0.981151,4.490112,LogisticRegression,<NA>
4,PyCaret,breast_cancer,123,0.973684,0.974106,0.969246,0.971583,5.586567,LogisticRegression,<NA>
5,PyCaret,breast_cancer,2026,0.973684,0.974106,0.969246,0.971583,6.672980,LogisticRegression,<NA>
6,TPOT,breast_cancer,42,0.964912,0.962302,0.962302,0.962302,612.866787,"Pipeline(steps=[('stackingestimator',\n ...",<NA>
7,TPOT,breast_cancer,123,0.964912,0.967230,0.957341,0.961911,602.783760,"Pipeline(steps=[('stackingestimator',\n ...",<NA>
8,TPOT,breast_cancer,2026,0.964912,0.962302,0.962302,0.962302,605.559059,"Pipeline(steps=[('standardscaler', StandardSca...",<NA>
9,H2O AutoML,titanic,42,0.824427,0.813603,0.815926,0.814710,590.980065,StackedEnsemble_BestOfFamily_6_AutoML_7_202607...,60


In [8]:
all_results.to_csv(
    RESULTS_DIR / "all_results.csv",
    index=False,
)

In [24]:
experiment_counts = (
    all_results
    .groupby(["framework", "dataset"])
    .size()
    .reset_index(name="number_of_runs")
)

display(experiment_counts)

,framework,dataset,number_of_runs
0,H2O AutoML,breast_cancer,3
1,H2O AutoML,titanic,3
2,H2O AutoML,wine,3
3,PyCaret,breast_cancer,3
4,PyCaret,titanic,3
5,PyCaret,wine,3
6,TPOT,breast_cancer,3
7,TPOT,titanic,3
8,TPOT,wine,3


In [10]:
overall_summary = (
    all_results
    .groupby("framework")[metrics]
    .agg(["mean", "std"])
    .round(4)
)

overall_summary.to_csv(
    RESULTS_DIR / "overall_summary.csv"
)

display(overall_summary)

accuracy         precision_macro         recall_macro          \
               mean     std            mean     std         mean     std   
framework                                                                  
H2O AutoML   0.8996  0.0858          0.8953  0.0917       0.9004  0.0892   
PyCaret      0.9161  0.0956          0.9137  0.1010       0.9079  0.1051   
TPOT         0.9056  0.0888          0.9033  0.0952       0.8983  0.0982   

           f1_macro         runtime_seconds           
               mean     std            mean      std  
framework                                             
H2O AutoML   0.8957  0.0901        598.2717   5.1400  
PyCaret      0.9100  0.1036          6.0148   0.8555  
TPOT         0.8998  0.0967        612.6823  13.8552

In [25]:
metrics = [
    "accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "runtime_seconds",
]

summary_by_dataset = (
    all_results
    .groupby(["dataset", "framework"])[metrics]
    .agg(["mean", "std"])
    .round(4)
)

summary_by_dataset.to_csv(
    RESULTS_DIR / "summary_by_dataset.csv"
)

display(summary_by_dataset)

accuracy         precision_macro          \
                             mean     std            mean     std   
dataset       framework                                             
breast_cancer H2O AutoML   0.9444  0.0415          0.9367  0.0456   
              PyCaret      0.9766  0.0051          0.9765  0.0041   
              TPOT         0.9649  0.0000          0.9639  0.0028   
titanic       H2O AutoML   0.7913  0.0289          0.7802  0.0295   
              PyCaret      0.7901  0.0238          0.7807  0.0263   
              TPOT         0.7888  0.0242          0.7779  0.0247   
wine          H2O AutoML   0.9630  0.0160          0.9690  0.0093   
              PyCaret      0.9815  0.0160          0.9840  0.0139   
              TPOT         0.9630  0.0160          0.9682  0.0138   

                         recall_macro         f1_macro          \
                                 mean     std     mean     std   
dataset       framework                                          
breast_cancer H2O AutoML       0.9527  0.0356   0.9420  0.0430   
              PyCaret          0.9732  0.0069   0.9748  0.0055   
              TPOT             0.9606  0.0029   0.9622  0.0002   
titanic       H2O AutoML       0.7866  0.0276   0.7820  0.0290   
              PyCaret          0.7697  0.0281   0.7735  0.0266   
              TPOT             0.7693  0.0301   0.7727  0.0282   
wine          H2O AutoML       0.9619  0.0247   0.9631  0.0194   
              PyCaret          0.9810  0.0172   0.9818  0.0159   
              TPOT             0.9651  0.0120   0.9647  0.0139   

                         runtime_seconds           
                                    mean      std  
dataset       framework                            
breast_cancer H2O AutoML        600.8749   2.4134  
              PyCaret             5.5832   1.0914  
              TPOT              607.0699   5.2085  
titanic       H2O AutoML        591.8526   2.3543  
              PyCaret             5.9604   0.7857  
              TPOT              626.5462  16.7093  
wine          H2O AutoML        602.0876   0.7049  
              PyCaret             6.5006   0.6935  
              TPOT              604.4308   4.8834

In [14]:
final_summary_table = mean_std_table(
    dataframe=all_results,
    group_columns=["dataset", "framework"],
    metrics=[
        "accuracy",
        "precision_macro",
        "recall_macro",
        "f1_macro",
        "runtime_seconds",
    ],
)

display(final_summary_table)

,dataset,framework,accuracy,precision_macro,recall_macro,f1_macro,runtime_seconds
0,breast_cancer,H2O AutoML,0.9444 ± 0.0415,0.9367 ± 0.0456,0.9527 ± 0.0356,0.942 ± 0.043,600.8749 ± 2.4134
1,breast_cancer,PyCaret,0.9766 ± 0.0051,0.9765 ± 0.0041,0.9732 ± 0.0069,0.9748 ± 0.0055,5.5832 ± 1.0914
2,breast_cancer,TPOT,0.9649 ± 0.0,0.9639 ± 0.0028,0.9606 ± 0.0029,0.9622 ± 0.0002,607.0699 ± 5.2085
3,titanic,H2O AutoML,0.7913 ± 0.0289,0.7802 ± 0.0295,0.7866 ± 0.0276,0.782 ± 0.029,591.8526 ± 2.3543
4,titanic,PyCaret,0.7901 ± 0.0238,0.7807 ± 0.0263,0.7697 ± 0.0281,0.7735 ± 0.0266,5.9604 ± 0.7857
5,titanic,TPOT,0.7888 ± 0.0242,0.7779 ± 0.0247,0.7693 ± 0.0301,0.7727 ± 0.0282,626.5462 ± 16.7093
6,wine,H2O AutoML,0.963 ± 0.016,0.969 ± 0.0093,0.9619 ± 0.0247,0.9631 ± 0.0194,602.0876 ± 0.7049
7,wine,PyCaret,0.9815 ± 0.016,0.984 ± 0.0139,0.981 ± 0.0172,0.9818 ± 0.0159,6.5006 ± 0.6935
8,wine,TPOT,0.963 ± 0.016,0.9682 ± 0.0138,0.9651 ± 0.012,0.9647 ± 0.0139,604.4308 ± 4.8834


In [15]:
final_summary_table.to_csv(
    RESULTS_DIR / "final_summary_table.csv",
    index=False,
)

In [27]:
f1_comparison = all_results.pivot_table(
    index="dataset",
    columns="framework",
    values="f1_macro",
    aggfunc=["mean", "std"],
).round(4)

f1_comparison.to_csv(
    RESULTS_DIR / "f1_comparison.csv"
)

display(f1_comparison)

mean                        std                
framework     H2O AutoML PyCaret    TPOT H2O AutoML PyCaret    TPOT
dataset                                                            
breast_cancer     0.9420  0.9748  0.9622     0.0430  0.0055  0.0002
titanic           0.7820  0.7735  0.7727     0.0290  0.0266  0.0282
wine              0.9631  0.9818  0.9647     0.0194  0.0159  0.0139

In [28]:
runtime_comparison = all_results.pivot_table(
    index="dataset",
    columns="framework",
    values="runtime_seconds",
    aggfunc=["mean", "std"],
).round(2)

runtime_comparison.to_csv(
    RESULTS_DIR / "runtime_comparison.csv"
)

display(runtime_comparison)

mean                        std               
framework     H2O AutoML PyCaret    TPOT H2O AutoML PyCaret   TPOT
dataset                                                           
breast_cancer     600.87    5.58  607.07       2.41    1.09   5.21
titanic           591.85    5.96  626.55       2.35    0.79  16.71
wine              602.09    6.50  604.43       0.70    0.69   4.88

In [29]:
print("Total rows:", len(all_results))
print("\nMissing values:")
print(all_results.isna().sum())

assert len(all_results) == 27
assert experiment_counts["number_of_runs"].eq(3).all()

print("\nAll 27 experiments are present and correctly grouped.")

Total rows: 27

Missing values:
framework            0
dataset              0
seed                 0
accuracy             0
precision_macro      0
recall_macro         0
f1_macro             0
runtime_seconds      0
best_model           0
models_evaluated    18
dtype: int64

All 27 experiments are present and correctly grouped.


In [18]:
experiment_counts = (
    all_results
    .groupby(["framework", "dataset"])
    .size()
    .reset_index(name="number_of_runs")
)

display(experiment_counts)

,framework,dataset,number_of_runs
0,H2O AutoML,breast_cancer,3
1,H2O AutoML,titanic,3
2,H2O AutoML,wine,3
3,PyCaret,breast_cancer,3
4,PyCaret,titanic,3
5,PyCaret,wine,3
6,TPOT,breast_cancer,3
7,TPOT,titanic,3
8,TPOT,wine,3


In [19]:
print("Total rows:", len(all_results))
print("Missing values by column:")
print(all_results.isna().sum())

Total rows: 27
Missing values by column:
framework            0
dataset              0
seed                 0
accuracy             0
precision_macro      0
recall_macro         0
f1_macro             0
runtime_seconds      0
best_model           0
models_evaluated    18
dtype: int64
